# Tomás Solano, InZenFenix, IIT414W, 22-03-2026

In [7]:
# ── Cell 1: Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings
import platform

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')
print("\nSystem: " + platform.system())
print("Distro: " + platform.release())

Python  : 3.14.3
NumPy   : 2.2.3
Seed    : 414

System: Linux
Distro: 6.19.9-1-cachyos


# 1. DATA, LIBRARIES AND SPLIT

In [8]:
# ──────────────────────────────────── DATA AND LIBRARIES ───────────────────────────────────────────────

import fastf1
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

data_df = pd.DataFrame({ "TeamName": [], "Classified Position": []})
race_nums = range(15,24)

for race_num in race_nums:
    session = fastf1.get_session(2024, race_num, 'R')
    session.load(laps=False, telemetry=False)
    
    results = session.results[['FullName', 'ClassifiedPosition', 'TeamName']]
    
    
    for _, row in results.iterrows():
        # Handle DNFs or unclassified if necessary
        # Also get the data as 0s and 1s to calculate accuracy
        data_df.loc[-1] = [row["TeamName"], row["ClassifiedPosition"]]
        data_df.index = data_df.index + 1

# ──────────────────────────────────── Training Data and Transformation ────────────────────────────────────

# 1 if is top 10 else is 0
#if its a digit, else its usually R or DNF, so goes to 0
def get_positions(x):
    if(str.isdigit(x)):
        return 1 if int(x) <= 10 else 0
    
    else:
        return 0
    
data_df["IsTop10"] = data_df["Classified Position"].apply(get_positions)
X = pd.get_dummies(data_df["TeamName"], drop_first=True, dtype=int)
y = data_df["IsTop10"]

# The data is organized as 24 race to 19 all in order, so we can split it to get data from different races
# and the remaining 33% (the most recent races) for testing.
# For example if we have 6 races, so it would be 4 races for training and 2 for testing
split_index = int(len(data_df) * 0.67)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

core           INFO 	Loading data for Dutch Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '16', '81', '55', '11', '63', '44', '10', '14', '27', '3', '18', '23', '31', '2', '22', '20', '77', '24']
core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '4', '55', '44', '1', '63', '11', '23', '20', '14', '43', '3', '31', '10', '77', '27', '24', '18', '22']
core           INFO 	Loading data for A

## 2. Majority-Class Model

In [ ]:
#Code obtained from W03_Wed_baselines_and_metrics.ipynb, modified to use the data from our dataset
# And added new set of scores

clf_majority = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
clf_majority.fit(X_train, y_train)
y_pred_majority = clf_majority.predict(X_test)

majority_class = clf_majority.classes_[np.argmax(clf_majority.class_prior_)]
majority_label = "Top-10" if majority_class == 1 else "Not Top-10"

print("=== Majority-Class Baseline ===")
print(f"What it predicts: always class {majority_class} ({majority_label})")
print(f"Class balance (train): {y_train.mean():.3f} Top-10 | {1 - y_train.mean():.3f} Not Top-10\n")

print(classification_report(
    y_test, y_pred_majority,
    target_names=['Not Top-10', 'Top-10'],
    zero_division=0,
))

print(f"Accuracy        : {accuracy_score(y_test, y_pred_majority):.4f}")
print(f"F1 (Top-10)     : {f1_score(y_test, y_pred_majority, pos_label=1):.4f}")
print(f"F1 (macro avg)  : {f1_score(y_test, y_pred_majority, average='macro'):.4f}")
print("Precision: ", precision_score(y_test, y_pred_majority))
print("Recall: ", recall_score(y_test, y_pred_majority))
print("ROC-AUC: ", roc_auc_score(y_test, y_pred_majority))

=== Majority-Class Baseline ===
What it predicts: always class 0 (Not Top-10)
Class balance (train): 0.500 Top-10 | 0.500 Not Top-10

              precision    recall  f1-score   support

  Not Top-10       0.50      1.00      0.67        30
      Top-10       0.00      0.00      0.00        30

    accuracy                           0.50        60
   macro avg       0.25      0.50      0.33        60
weighted avg       0.25      0.50      0.33        60

Accuracy        : 0.5000
F1 (Top-10)     : 0.0000
F1 (macro avg)  : 0.3333
Precision:  0.0
Recall:  0.0
Recall:  0.5


/home/inzenfenix/Documents/GitHub/iit414w-lab01-STS17/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
